# Lab — Synthetic Trajectory Generation + Photoreal Augmentation
### Isaac Sim · Isaac Lab Mimic · Cosmos Transfer 2.5 NIM

This notebook walks you through the **data-generation half** of an augmented
imitation-learning pipeline. By the end you will have:

1. **Synthesized** a brand-new robot manipulation trajectory in Isaac Sim
   from a small seed dataset (10 human demos) using Isaac Lab Mimic.
2. **Encoded** that trajectory as a hybrid *shaded-segmentation* control video
   (semantic segmentation × surface-normal Lambertian shading).
3. **Augmented** the trajectory through **NVIDIA Cosmos Transfer 2.5 NIM** —
   re-rendering the same robot motion as a photoreal video under any text
   prompt you compose ("rustic copper cubes in a university lab", "marble
   cubes in a high-tech cleanroom", etc.).

```
   10 human demos                                          photoreal
        │                                              MP4 you can watch
        ▼ ┌──────── Stage 1: Mimic ────────┐             & train on
          │ Spatial transform + replay     │
          │ → physically-valid trajectory  │
          └────────────┬───────────────────┘
                       │ shaded-segmentation MP4
                       ▼ ┌── Stage 2: Cosmos T2.5 NIM ──┐
                         │ Diffusion control-net + LLM  │ ─── photoreal MP4 ───►
                         │ Same motion, any visual look │
                         └──────────────────────────────┘
```

The two halves run in **separate Docker containers** (`smmg-lab` and
`cosmos-nim`) that talk over HTTP. The notebook orchestrates both.

> **NOTE:** Run the cells top-to-bottom. The first run of Cell 2 (Isaac Sim
> boot) takes ~3-5 minutes — Vulkan is compiling shaders and the USD scene
> is loading. Subsequent cells are fast.


# What's actually happening under the hood

This lab uses three distinct NVIDIA technologies. Knowing what each does
makes the rest of the cells make sense.

## Stage 1 — Isaac Lab Mimic

**Mimic** is a trajectory-multiplication tool bundled with Isaac Lab. Given:

- a small seed dataset of human teleoperated demos (~10 trajectories),
  each annotated with **subtask boundaries** (e.g. "approach cube 1",
  "grasp", "lift", "place on cube 2"),
- a randomized scene (cubes in different positions, arm starting from
  different poses),

…Mimic spatially transforms the source subtask segments to land each
sub-goal at the new randomized pose, stitches them back together, replays
the result in Isaac Sim physics, and keeps the runs that succeed. Net effect:
**10 demos → unlimited synthetic trajectories.**

The seed dataset for this lab is `datasets/annotated_dataset.hdf5` — the same
one NVIDIA published with the Synthetic Manipulation Motion Generation
blueprint.

## Stage 2 — Cosmos Transfer 2.5

**Cosmos Transfer 2.5** is NVIDIA's controlnet-style video diffusion model.
Given:

- an **input video** (used as a control signal — edges, depth, segmentation,
  or blur),
- a **text prompt** describing the desired look,

…it generates a new video that follows the *geometry* of the control video
but takes its *appearance* from the prompt. **Same motion, any visual look.**

We feed Cosmos a *shaded-segmentation* video — flat semantic segmentation
colors multiplied by Lambertian shading from the surface normals. This
hybrid has crisp object edges (so Cosmos preserves geometry) and 3D shape
cues (so depth survives), but no real-world texture (so the prompt is
free to drive the look).

## Stage 3 — NVIDIA NIM

The Cosmos Transfer 2.5 model is hosted as a **NIM** (NVIDIA Inference
Microservice): a Triton-backed container with persistent in-memory model
weights and standard HTTP endpoints. Submitting an inference is a single
`POST /v1/infer` returning the photoreal video inline as base64.

We run NIM on a separate Hopper / Blackwell host (e.g. an H200) and this
notebook submits to it over the network.


---

# Step 1 — Generate a new motion trajectory with Mimic

The next four cells:

| Cell | What it does |
|---|---|
| **Configure trials** | How many successful trajectories to synthesize this run |
| **Spin up Isaac Sim** | Boots Omniverse Kit headless, loads the Franka stacking scene, creates the Mimic env |
| **Randomize sliders** | Pick how much to vary cube positions and Franka starting pose between trials |
| **Run Mimic** | The actual synthesis loop — async coroutines push subtask-stitched actions, the env steps and records |

Outputs land in `datasets/generated_dataset.hdf5` (low-dim states + actions)
and `_isaaclab_out/*.png` (per-frame camera dumps for the shaded-segmentation
encoder in Step 2).


## Cell 1 — Configure number of trials

Sets the target count of *successful* trials Mimic should produce. The
simulator may attempt more than this — failed runs (cube knocked off table,
gripper missed, etc.) don't count.

**Recommended values:**
- `1` for a first end-to-end sanity check (~30-60 s).
- `5-20` for a real fine-tune dataset (each trial gives a unique trajectory
  to augment with Cosmos in Step 2).

The widget below stores its value in `num_trials.value`, read by the next
cell. Adjust the number, then move on.


In [1]:
from notebook_widgets import create_num_trials_input

num_envs = 1
num_trials = create_num_trials_input()


HTML(value='<h3>Set Number of Trials</h3>')

HTML(value='<p>How many demonstrations to generate</p>')

BoundedIntText(value=1, description='Number of Trials:', layout=Layout(width='300px'), min=1, style=Descriptio…

## Cell 2 — Spin up Isaac Sim and create the Mimic env

This cell does the heaviest one-time work in the whole notebook:

1. **Boots Omniverse Kit** with the headless rendering experience file
   (`isaaclab.python.headless.rendering.kit`). Vulkan compiles shaders the
   first time — that's why this takes ~3-5 minutes initially.
2. **Loads the cube-stacking scene** — a Franka Panda arm, three colored
   cubes, table, and lighting (USD scene graph).
3. **Constructs the Mimic env** (`FrankaCubeStackIKRelBlueprintMimicEnvCfg`)
   which knows how to spatially transform the seed demos to fit the
   randomized scene.

> **NOTE:** You may briefly see warnings like
> `[Warning] [simulation_app] Non-headless mode not supported with jupyter notebooks`
> and `[Warning] [carb.windowing-glfw] GLFW initialization failed.` —
> these are informational; we *want* headless mode here.

> **NOTE 2:** The cell forces `args_cli.headless = True` explicitly. Without
> it, AppLauncher loads the wrong experience file and camera buffers don't
> refresh between sim steps — so PNG dumps would be stationary even though
> the env is moving.

**Expected end of cell output:**
```
[INFO]: Time taken for scene creation : ~3-5 seconds
| Driver Version: ...     | Graphics API: Vulkan
| GPU 0: NVIDIA RTX PRO 6000 / H100 / H200 / ...
[INFO] Action Manager: <ActionManager> contains 2 active terms.
[INFO] Observation Manager: <ObservationManager> contains 3 groups.
[INFO]: Completed setting up the environment...
```

After this completes, the global variable `env` holds the live Mimic env.


In [2]:
import os
import nest_asyncio
nest_asyncio.apply()

from argparse import ArgumentParser, Namespace
from isaaclab.app import AppLauncher

parser = ArgumentParser()
AppLauncher.add_app_launcher_args(parser)
args_cli = parser.parse_args([])
args_cli.enable_cameras = True
args_cli.headless = True
args_cli.kit_args = "--enable omni.videoencoding"

config = {
    "task": "Isaac-Stack-Cube-Franka-IK-Rel-Blueprint-Mimic-v0",  
    "num_envs": num_envs,                                       
    "generation_num_trials": num_trials.value,                         
    "input_file": "datasets/annotated_dataset.hdf5",     
    "output_file": "datasets/generated_dataset.hdf5", 
    "pause_subtask": False,
    "enable": "omni.kit.renderer.capture",
}

# Update the default configuration
args_dict = vars(args_cli)
args_dict.update(config)
args_cli = Namespace(**args_dict)

# Now launch the simulator with the final configuration
app_launcher = AppLauncher(args_cli)
simulation_app = app_launcher.app

import asyncio
import gymnasium as gym
import numpy as np
import random
import torch

import isaaclab_mimic.envs  # noqa: F401
from isaaclab_mimic.datagen.generation import env_loop, setup_env_config, setup_async_generation
from isaaclab_mimic.datagen.utils import get_env_name_from_dataset, setup_output_paths, interactive_update_randomizable_params, reset_env
from isaaclab.managers import ObservationTermCfg as ObsTerm
from notebook_utils import ISAACLAB_OUTPUT_DIR

import isaaclab_tasks  # noqa: F401
num_envs = args_cli.num_envs

# Setup output paths and get env name
output_dir, output_file_name = setup_output_paths(args_cli.output_file)
env_name = args_cli.task or get_env_name_from_dataset(args_cli.input_file)

# Configure environment
env_cfg, success_term = setup_env_config(
    env_name=env_name,
    output_dir=output_dir,
    output_file_name=output_file_name,
    num_envs=num_envs,
    device=args_cli.device,
    generation_num_trials=args_cli.generation_num_trials,
)
# Set observation output directory
for obs in vars(env_cfg.observations.rgb_camera).values():
    if not isinstance(obs, ObsTerm):
        continue
    obs.params["image_path"] = os.path.join(ISAACLAB_OUTPUT_DIR, obs.params["image_path"])
env_cfg.observations


# create environment
env = gym.make(env_name, cfg=env_cfg).unwrapped

# set seed for generation
random.seed(env.cfg.datagen_config.seed)
np.random.seed(env.cfg.datagen_config.seed)
torch.manual_seed(env.cfg.datagen_config.seed)

# reset before starting
reset_env(env, 100)


[WARN][AppLauncher]: There are no arguments attached to the ArgumentParser object. If you have your own arguments, please load your own arguments before calling the `AppLauncher.add_app_launcher_args` method. This allows the method to check the validity of the arguments and perform checks for argument names.
[INFO][AppLauncher]: Loading experience file: /workspace/isaaclab/apps/isaaclab.python.rendering.kit
[Warning] [simulation_app] Interactive python shell detected but ISAAC_JUPYTER_KERNEL was not set. Problems with asyncio may occur
[Warning] [simulation_app] Please use Isaac Sim Python 3 kernel instead of the default Python 3 Kernel
[Info] [carb] Logging to file: /isaac-sim/kit/logs/Kit/Isaac-Sim/4.5/kit_20260426_082334.log
2026-04-26 08:23:34 s] [Warning] [omni.kit.app.plugin] No crash reporter present, dumps uploading isn't available.
2026-04-26 08:23:43 [8,789ms] [Warning] [omni.datastore] OmniHub is inaccessible
2026-04-26 08:24:02 [28,069ms] [Warning] [carb.windowing-glfw.game

/isaac-sim/exts/omni.isaac.ml_archive/pip_prebundle/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA RTX PRO 6000 Blackwell Server Edition with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_37 sm_90.
If you want to use the NVIDIA RTX PRO 6000 Blackwell Server Edition GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


[INFO]: Parsing configuration from: <class 'isaaclab_mimic.envs.franka_stack_ik_rel_blueprint_mimic_env_cfg.FrankaCubeStackIKRelBlueprintMimicEnvCfg'>
[INFO]: Base environment:
	Environment device    : cuda:0
	Environment seed      : None
	Physics step-size     : 0.01
	Rendering step-size   : 0.05
	Environment step-size : 0.05
[INFO]: Time taken for scene creation : 1.773312 seconds


## Cell 3 — Randomize per-trial scene parameters

Each Mimic trial starts from a slightly different scene. The widgets below
let you tune the randomization range:

| Parameter | What it controls |
|---|---|
| `randomize_franka_joint_state.mean` | Average joint-angle offset added to the canonical start pose, in radians (0 = always start from canonical) |
| `randomize_franka_joint_state.std` | Std-dev of per-joint Gaussian noise on top of the mean |
| `randomize_cube_positions.pose_range.x` | x-range (along table depth) where the three cubes spawn, in meters |
| `randomize_cube_positions.pose_range.y` | y-range (across table) where they spawn |
| `randomize_cube_positions.min_separation` | Minimum distance between any two cubes |

**Recommended for first run:** accept the defaults — they sit comfortably
inside what the seed demos can spatially transform to reach.

If Mimic later reports `0/N successful demos`, **tighten** these ranges
(e.g. `pose_range.x` 0.45-0.55, `pose_range.y` -0.05 to 0.05) so cubes spawn
closer to the locations the source demos handle — then re-run Cell 4.

> **NOTE:** Isaac Lab 2.2.1 added a new reset event `init_franka_arm_pose`
> that doesn't have a slider in this dict. We silently skip events we don't
> have parameters for — that's expected.


In [ ]:
randomizable_params = {
    "randomize_franka_joint_state": {
        "mean": (0.0, 0.5, 0.01),
        "std": (0.0, 0.1, 0.01),
    },
    "randomize_cube_positions": {
        "pose_range": {
                "x": (0.3, 0.9, 0.01),
                "y": (-0.3, 0.3, 0.01),
            },
        "min_separation": (0.0, 0.5, 0.01),
    }
}

for i in range(len(env.unwrapped.event_manager._mode_term_cfgs["reset"])):
    event_term = env.unwrapped.event_manager._mode_term_cfgs["reset"][i]
    name = env.unwrapped.event_manager.active_terms["reset"][i]
    display(f"Updating parameters for event: {event_term.func.__name__}")
    if name not in randomizable_params:
        continue
    interactive_update_randomizable_params(event_term, name, randomizable_params[name], env=env)


## Cell 4 — Run Mimic synthesis

This is where the action happens. Two coroutines run concurrently:

- **Generator** — pulls a source demo from the seed dataset, spatially
  transforms each subtask segment to land at the new randomized pose,
  pushes the resulting actions onto an `asyncio.Queue`.
- **Executor** (`env_loop`) — drains actions from the queue, calls
  `env.step(...)` to advance physics, records observations + actions to
  HDF5, and signals success / failure via the success-termination function.

The output you want to see:

```
Loaded 10 to datagen info pool
**************************************************
1/N (X%) successful demos generated by mimic
**************************************************
Reached 1 successes/attempts. Exiting.
Tasks were properly cancelled during cleanup.
```

`1/N successful demos` means **at least one Mimic trajectory worked** —
that's all you need to proceed. Stationary at `0/N` for many attempts
means the cubes are landing somewhere the source demos can't reach;
go back to Cell 3 and tighten the ranges.

**What ends up on disk:**
- `datasets/generated_dataset.hdf5` — 1+ trajectory (actions + low-dim states + per-frame pose)
- `_isaaclab_out/<camera>_<modality>_trial_<N>_tile_<E>_step_<F>.png` —
  ~250 PNGs per trial × 4 camera-modality streams (table_cam normals,
  table_cam segmentation, table_high_cam normals, table_high_cam segmentation)

Step 2 (Cosmos) consumes the segmentation + normals PNGs.

> **NOTE:** `env_loop` in IsaacLab 2.2.1 takes 5 positional args
> `(env, reset_queue, action_queue, info_pool, event_loop)` — one more
> than 2.0.2. The cell is updated for that.


In [ ]:
import sys
from IPython.display import display, Pretty

# Create a new output capture
class OutputCapture:
    def __init__(self):
        self._buffer = ""
    
    def write(self, text):
        if text.strip():  # Only process non-empty strings
            display(Pretty(text.rstrip()))
    
    def flush(self):
        if self._buffer:
            display(Pretty(self._buffer))
            self._buffer = ""

# Move stdout redirection before setup_async_generation
old_stdout = sys.stdout
sys.stdout = OutputCapture()

try:
    # Setup and run async data generation
    async_components = setup_async_generation(
        env=env,
        num_envs=args_cli.num_envs,
        input_file=args_cli.input_file,
        success_term=success_term,
        pause_subtask=args_cli.pause_subtask
    )

    future = asyncio.ensure_future(asyncio.gather(*async_components['tasks']))
    env_loop(env, async_components['reset_queue'], async_components['action_queue'], 
            async_components['info_pool'], async_components['event_loop'])
except asyncio.CancelledError:
    display(Pretty("Tasks were cancelled."))
except AttributeError as e:
    if "'FrankaCubeStackIKRelMimicEnv' object has no attribute 'scene'" in str(e):
        display(Pretty("Environment was closed during execution. This is expected behavior."))
except Exception as e:
    display(Pretty(f"Error occurred: {str(e)}"))
finally:
    # Restore original stdout first
    sys.stdout = old_stdout
    
    # Cancel the future and ignore any AttributeErrors from pending tasks
    if 'future' in locals():
        future.cancel()
        try:
            async_components['event_loop'].run_until_complete(future)
        except (asyncio.CancelledError, AttributeError) as e:
            if isinstance(e, AttributeError) and "'FrankaCubeStackIKRelMimicEnv' object has no attribute 'scene'" in str(e):
                display(Pretty("Environment was closed during execution. This is expected behavior!"))
            elif isinstance(e, asyncio.CancelledError):
                display(Pretty("Tasks were properly cancelled during cleanup."))
            else:
                display(Pretty(f"Unexpected cleanup error: {str(e)}"))


---

# Step 2 — Augment the trajectory photoreal with Cosmos Transfer 2.5

We now turn the simulated trajectory into a photoreal video. Two phases:

| Phase | What it does | Cells |
|---|---|---|
| **Pre-process** | Pick a camera, fuse segmentation+normals into a *shaded-segmentation* MP4 — Cosmos's preferred control input | Cells 5-6 |
| **Augment via NIM** | Compose a prompt, dial in T2.5 parameters, POST to NIM, receive photoreal MP4 inline | Cells 7-9 |

## Cell 5 — Pick a camera

The Mimic env captures from two cameras (`table_cam`, `table_high_cam`).
Pick which one you want as the source of the augmented video. **Use
`table_cam`** — it's the camera the published Robomimic policies were
trained on, so any later fine-tune lines up.

`VIDEO_LENGTH = 120` selects the last 120 frames of the trial (the most
action-dense segment, typically the final cube-placing motion).

## Cell 6 — Encode the shaded-segmentation MP4

For each frame, this cell:

1. Loads the segmentation PNG (flat colors per object) onto GPU
2. Loads the surface-normals PNG onto GPU
3. Runs a Warp kernel: `shade = 0.5 + 0.5 · dot(normal, light_dir)`,
   multiplies the segmentation colors by `shade`
4. Encodes the resulting frames as h264 MP4 via `imageio` (libx264 software
   — replaces the original NVENC path which fails on Blackwell with
   `NV_ENC_ERR_INVALID_PARAM`)

**Expected outcome:** an inline 5-second video showing a cel-shaded clip —
flat colored table + cubes + arm with crisp edges and visible depth shading.
This file (`_isaaclab_out/shaded_segmentation_*.mp4`) is the input we send
to Cosmos.


In [ ]:
from notebook_widgets import create_camera_input
from notebook_utils import ISAACLAB_OUTPUT_DIR

VIDEO_LENGTH = 120   # Suggested length is between 120 and 200
camera_selection = create_camera_input(ISAACLAB_OUTPUT_DIR)


In [ ]:
import os
from IPython.display import Video
from notebook_utils import encode_video, ISAACLAB_OUTPUT_DIR, get_env_trial_frames

env_trial_frames = get_env_trial_frames(ISAACLAB_OUTPUT_DIR, camera_selection.value, 10)
camera = camera_selection.value
for env_num, trial_nums in env_trial_frames.items():
    for trial_num, (start_frame, end_frame) in trial_nums.items():
        trial_length = end_frame - start_frame + 1
        if trial_length < VIDEO_LENGTH:
            print(f"\nSkipping Trial {trial_num}: Too short ({trial_length} frames)")
            continue
            
        video_start = max(start_frame, end_frame - VIDEO_LENGTH + 1)
        
        # Generate video filename with trial number
        video_filepath = os.path.join(ISAACLAB_OUTPUT_DIR, f"shaded_segmentation_{camera}_trial_{trial_num}_tile_{env_num}.mp4")
            
        try:
            encode_video(ISAACLAB_OUTPUT_DIR, video_start, VIDEO_LENGTH, 
                        camera, video_filepath, env_num, trial_num)
            display(video_filepath)
            display(Video(video_filepath, width=1000))
        except ValueError as e:
            print(f"Error processing trial {trial_num}: {str(e)}")


## Cell 7 — Point at the Cosmos Transfer 2.5 NIM

The notebook talks to NIM over HTTP. The URL widget below is pre-filled
from the `COSMOS_API_URL` env var (set to `http://cosmos-nim:8000` in
docker-compose, or whatever the NIM host's IP is for two-machine deploys).

**The widget fires a `/v1/health/ready` probe** as soon as it renders, so
you'll immediately see whether NIM is reachable from this kernel.

Healthy response:
```
Cosmos NIM health: {'object': 'Triton readiness check', 'message': 'ready', 'status': 'ready'}
```

If it errors:

| Symptom | Likely cause |
|---|---|
| `Connection refused` | NIM container is down — `docker ps` to check |
| `Connection timed out` | Network unreachable from this container — port 8000 not forwarded, or firewall |
| `Status: not ready` | NIM still loading models (first-time start takes 5-10 min while it pulls ~30 GB of weights) |

Edit the widget URL if needed, then re-run the cell.


In [ ]:
import ipywidgets as widgets
import os

# Default points at the NVIDIA Cosmos Transfer 2.5 NIM (port 8000).
# Override at the Jupyter prompt if NIM lives on a different host.
default_url = os.environ.get("COSMOS_API_URL", "http://cosmos-nim:8000")
url_widget = widgets.Text(
    value=default_url,
    placeholder="http://host:port",
    description="Cosmos NIM URL:",
    style={"description_width": "initial"},
    layout={"width": "700px"},
)
display(url_widget)

# Quick liveness check
try:
    from cosmos_t25_client import healthz
    print("Cosmos NIM health:", healthz(url_widget.value))
except Exception as e:
    print(f"(Cosmos NIM unreachable yet — set the URL above and re-run this cell): {e}")


## Cell 8 — Compose your prompt + Cosmos parameters

Two widget groups:

### Prompt mixer
A live-updated text prompt assembled from three dropdowns:

- **Cube material** — `glass`, `iridescent metal`, `marble`, `wood`,
  `cheese`, `origami paper`, `ice`, etc.
- **Table material** — `marble`, `concrete`, `carbon fiber`, …
- **Location** — `cluttered workshop`, `high-tech cleanroom`, `restaurant`,
  `university laboratory`, …

The combined prompt looks something like:
> "The scene depicts a robotic arm performing a precise block-stacking
> operation with **glass** cubes in a **high-tech cleanroom**. The
> workspace is centered on a **marble** table…"

### T2.5 inference parameters
Maps directly to NIM's `Transfer2Request` schema:

| Param | Range | Meaning |
|---|---|---|
| `seed` | int | Reproducibility — same seed + same prompt + same input → same output |
| `guidance` | 0-7 | Classifier-free guidance scale. Higher = stronger prompt adherence |
| `num_steps` | 4-80 | Diffusion sampling steps. 35 = published quality, 16 = faster, 8 = quick preview |
| `sigma_max` | 0-80 | Max noise level. Lower = closer to input control video, higher = more prompt expression |
| `resolution` | 256/480/512/720 | Output resolution. 720 = max quality, 480 = ~3× faster |
| `edge.control_weight` | 0-1 | How strongly Canny edges constrain the output (0 disables) |
| `depth.control_weight` | 0-1 | Depth control strength |
| `seg.control_weight` | 0-1 | Semantic segmentation control strength |
| `vis.control_weight` | 0-1 | Blur/visibility control strength |

**Recommended starting point for first run:**
- `seed=2025`, `guidance=3`, `num_steps=35`, `sigma_max=70`, `resolution=720`
- `edge.control_weight=1.0`, all other control weights `0`

Pick values, then move to the submit cell.


In [ ]:
from notebook_widgets import create_variable_dropdowns
from notebook_widgets_t25 import create_t25_params
from notebook_utils import ISAACLAB_OUTPUT_DIR

# Prompt mixer (cube × table × location) — reuse the original templates.
prompt_manager = create_variable_dropdowns("stacking_prompt.toml")

# Cosmos Transfer 2.5 inference parameters.
cosmos_params = create_t25_params(ISAACLAB_OUTPUT_DIR)
for w in cosmos_params.values():
    display(w)


## Cell 9 — Submit to Cosmos NIM and display result

This is the long cell. The client:

1. Reads the shaded-segmentation MP4 from disk
2. Base64-encodes it, builds a JSON request matching NIM's `Transfer2Request`
   schema (prompt, video, controls, seed, guidance, num_steps, sigma_max,
   resolution, num_conditional_frames)
3. **Synchronously POSTs** to `<NIM URL>/v1/infer` and **blocks** waiting
   for the response (NIM is sync — no submit/poll dance)
4. Decodes the returned `b64_video` field, writes the MP4 to
   `_cosmos_out/cosmos_t25_seed<N>.mp4`
5. Inline-plays the result via `IPython.display.Video`

### Time budget

| Settings | Approx wall-clock |
|---|---|
| `num_steps=35`, `resolution=720`, 120 frames | **~12-14 minutes** (chunked diffusion across 2 chunks) |
| `num_steps=16`, `resolution=720` | ~6-8 minutes |
| `num_steps=8`, `resolution=480` | ~2-3 minutes (quick preview quality) |

The cell appears "stuck" during diffusion — that's normal. If you watch
the NIM container's logs (`docker logs -f cosmos-nim` on the NIM host)
you'll see the diffusion progress bar tick through `35/35` per chunk.

### Expected output

```
[cosmos] POST <url>/v1/infer  body={'prompt': '...', 'video': '<b64 video, ~570 KB>',
       'resolution': '720', 'guidance': 3, 'num_steps': 35, ...}
[cosmos] saved → _cosmos_out/cosmos_t25_seed2025.mp4 (...bytes, ...s, seed=...)
```

Plus an inline video player showing your photoreal scene with the same
robot motion. **That's the demo asset.**

### Troubleshooting

| Symptom | Likely cause + fix |
|---|---|
| `RuntimeError: NIM /v1/infer returned 500` after a few seconds | NIM-side OOM or crash. Check H200 NIM logs; `docker restart cosmos-nim` if needed |
| `RuntimeError: NIM /v1/infer returned 400` | Bad parameter. Check the `body=` line in the cell output |
| Cell hangs >20 min | Diffusion in flight — check `docker logs -f cosmos-nim` for the progress bar |
| Inline video shows no player | VP9 codec compatibility issue with some browsers; the file IS on disk, open it in VLC |


In [ ]:
import os
from cosmos_t25_client import transfer
from notebook_widgets_t25 import widgets_to_t25_call_kwargs
from notebook_utils import ISAACLAB_OUTPUT_DIR, COSMOS_OUTPUT_DIR
from IPython.display import Video, clear_output

if not url_widget.value:
    raise ValueError("Cosmos API URL is empty.")

call = widgets_to_t25_call_kwargs(cosmos_params)
input_video_name = call.pop("_input_video_filename")
video_filepath = os.path.join(ISAACLAB_OUTPUT_DIR, input_video_name)

os.makedirs(COSMOS_OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(COSMOS_OUTPUT_DIR, f"cosmos_t25_seed{call['seed']}.mp4")

result = transfer(
    api_url=url_widget.value,
    video_path=video_filepath,
    output_path=output_path,
    prompt=prompt_manager.prompt,
    **call,
)

clear_output(wait=True)
print(f"Done — job_id={result['job_id']}")
print(f"Saved → {result['output_path']}")
display(Video(result["output_path"], width=900))


---

# What's next

You now have a working end-to-end pipeline:

```
seed (10 demos)            Mimic              Cosmos T2.5 NIM
        │              generates new        rerenders trajectory
        │              trajectory in        as photoreal video
        ▼                                                   ▼
   annotated_dataset →  generated_dataset.hdf5  →  cosmos_t25_seedN.mp4
   .hdf5                +  shaded_segmentation_*    (and as many more
   (input)                 .mp4 (intermediate)       as you want)
```

To scale into a fine-tune-quality dataset:

- **Re-run cells 1-4** with a higher trial count (e.g. 10) and varied
  randomization sliders → multiple Mimic trajectories.
- **For each trajectory, re-run cells 5-9** with different prompt
  combinations (cube × table × location) → multiple photoreal variants
  per trajectory.
- A small fine-tune set is typically **~10-30 photoreal clips** total.
  At ~10 min/clip on a single H200, that's a 2-5 hour batch you can
  leave running unattended.

**Part 2** of this lab covers:
- A Python batch driver that fans out (input × prompt) pairs against NIM
- Fine-tuning the NVIDIA-published Robomimic Policy B (epoch_600) on the
  augmented dataset
- Robust evaluation across 6 visual perturbation settings — vanilla,
  light intensity / color / texture, table texture, robot arm texture
- Side-by-side demo videos comparing baseline / NVIDIA-augmented / our
  fine-tuned policies

For now: pat yourself on the back. You've gone from 10 human demos →
synthetic trajectory → photoreal video, all driven from one notebook.
